# Aligning two sections in the plane

Everything the volume notebooks do, one rank down and without an atlas: two sections of the
same tissue brought into a common frame. Three routes, on the same pair of MERFISH sections,
in increasing order of what they can express:

| route | fits | when it is enough |
| --- | --- | --- |
| `align_landmarks` | a closed-form affine from paired points | the sections differ by a rigid move and a scale |
| `align_stalign_obs` | a diffeomorphism between two point clouds | tissue has stretched or torn |
| `align_stalign_image` | a diffeomorphism between two images | one side is an image, not points |

The fourteen rank-2 notebooks upstream ships are these three routes with different files.

## Inputs

Two MERFISH replicate sections, and thirteen landmark pairs picked on them by hand. The
landmarks are stored as `(x, y)` -- which is what squidpy's public API takes, so unlike
upstream's own notebooks nothing here transposes them on the way in.

In [ ]:
import anndata as ad, numpy as np, pandas as pd

MERFISH = 'merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate'

def section(replicate):
    """One replicate's centroids as an AnnData, coordinates in microns."""
    df = pd.read_csv(f'{MERFISH}{replicate}_cell_metadata_S2R{replicate}.csv.gz')
    xy = np.c_[df['center_x'], df['center_y']].astype(float)
    return ad.AnnData(X=np.zeros((len(xy), 1)), obsm={'spatial': xy})

ref, query = section(2), section(3)                      # S2R2 is the reference, S2R3 moves

landmarks = {r: np.asarray(np.load(f'merfish_data/Merfish_S2_R{r}_points.npy',
                                   allow_pickle=True).item()['all'], dtype=float)
             for r in (2, 3)}
print(f'{ref.n_obs} reference cells, {query.n_obs} query cells, '
      f'{len(landmarks[2])} landmark pairs')

## 1. Landmarks alone

`align_landmarks` solves for the affine in closed form -- no iteration, no images. `fit`
chooses how much freedom it gets: `"similarity"` allows rotation, one scale and translation;
`"affine"` adds non-uniform scale and shear. The constrained fit is the safer default because
it cannot shear a section that should not be sheared.

In [ ]:
from squidpy.experimental.tl import align_landmarks

query.obsm['landmarks'] = landmarks[3]
ref.obsm['landmarks'] = landmarks[2]

affine = align_landmarks(ref, query, landmark_key='landmarks', fit='affine')
print(np.asarray(affine).round(3))

residual = np.linalg.norm(
    landmarks[3] @ np.asarray(affine)[:2, :2].T + np.asarray(affine)[:2, 2] - landmarks[2],
    axis=1)
print(f'landmark residual: median {np.median(residual):.1f} um, worst {residual.max():.1f} um')

## 2. Two point clouds

`align_stalign_obs` rasterizes both sides internally and fits a diffeomorphism. Handing it the
same landmarks does two separate things: they derive the starting affine, and they stay in the
objective as a matching term, so the fit is pulled toward them rather than merely started there.

`dx` and `blur` are the rasterization; upstream's own notebook uses 30 um and 1.5.

In [ ]:
from squidpy.experimental.tl import align_stalign_obs

fit = align_stalign_obs(
    ref, query, spatial_key='spatial',
    landmarks_ref=landmarks[2], landmarks_query=landmarks[3],
    dx=30.0, blur=1.5, niter=10000, epV=50,
)
print(f'{fit.n_iter} iterations, objective '
      f'{float(fit.energies[0]):.0f} -> {float(fit.energies[-1]):.0f}')

Where the query cells end up. `transform` evaluates the fitted map at each point, so a cell
lands where it lands rather than at the nearest raster cell.

In [ ]:
import matplotlib.pyplot as plt

moved = np.asarray(fit.transform(query.obsm['spatial']))
started = np.asarray(align_landmarks(ref, query, landmark_key='landmarks', fit='affine'))
affine_only = query.obsm['spatial'] @ started[:2, :2].T + started[:2, 2]

fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))
for a, (pts, title) in zip(ax, [
        (query.obsm['spatial'], 'before'),
        (affine_only, 'after the landmark affine'),
        (moved, 'after the diffeomorphism')], strict=True):
    a.scatter(*ref.obsm['spatial'].T, s=0.6, alpha=0.15, label='reference (S2R2)')
    a.scatter(*pts.T, s=0.6, alpha=0.15, label='query (S2R3)')
    a.set_title(title); a.set_aspect('equal'); a.invert_yaxis()
    a.set_xticks([]); a.set_yticks([])
ax[0].legend(markerscale=20, loc='lower left', fontsize=8)

## 3. Points onto an image

When one side is an image there is nothing to rasterize it into, so the *other* side is
rasterized instead and `align_stalign_image` fits image to image. The two live in different
units -- microns for the MERFISH section, pixels for the Visium H&E -- and neither is restated
anywhere: each element carries its own placement and the solver reads the units off it.

In [ ]:
import matplotlib.pyplot as plt
import spatialdata as sd
from spatialdata.models import Image2DModel, PointsModel
from spatialdata.transformations import Scale
from squidpy.experimental.im import rasterize_points
from squidpy.experimental.tl import align_stalign_image

he = plt.imread('visium_data/tissue_hires_image.png')[..., :3]

visium = sd.SpatialData(images={'he': Image2DModel.parse(
    np.moveaxis(he, -1, 0).astype(float), dims=('c', 'y', 'x'))})

merfish = sd.SpatialData(points={'cells': PointsModel.parse(query.obsm['spatial'])})
rasterize_points(merfish, 'cells', dx=30.0, blur=1.0, key_added='section')

# Paired landmarks again, but picked on *these* two -- five named regions, in each side's own
# units. Row order is the correspondence, so both sides are flattened the same way.
picked = {side: np.load(f'visium_data/{name}_points.npy', allow_pickle=True).item()
          for side, name in (('query', 'Merfish_S2_R3'), ('ref', 'tissue_hires_image'))}
regions = list(picked['ref'])
paired = {side: np.array([p for r in regions for p in picked[side][r]], dtype=float)
          for side in picked}
print(f'{len(paired["ref"])} landmark pairs over {len(regions)} regions: {", ".join(regions)}')

In [ ]:
image_fit = align_stalign_image(
    visium, merfish, image_key=('he', 'section'),
    landmarks_ref=paired['ref'], landmarks_query=paired['query'],
    niter=200, sigmaM=0.2, sigmaB=0.19, sigmaA=0.3, sigmaP=2e-1,
    epL=5e-11, epT=5e-4, epV=5e1,
)
print(f'{image_fit.n_iter} iterations, objective '
      f'{float(image_fit.energies[0]):.0f} -> {float(image_fit.energies[-1]):.0f}')

In [ ]:
placed = np.asarray(image_fit.transform(query.obsm['spatial']))
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(he)
ax[0].scatter(*paired['ref'].T, s=25, c='red', label='target landmarks')
ax[0].set_title('Visium H&E'); ax[0].legend(fontsize=8)
ax[1].imshow(he)
ax[1].scatter(*placed.T, s=0.4, alpha=0.15, c='tab:blue')
ax[1].set_title('MERFISH cells placed on it')
for a in ax:
    a.set_xticks([]); a.set_yticks([])

## What this covers

`align_stalign_obs` is the shape of ten upstream notebooks -- the four `merfish-merfish`
variants, `merfish-xenium`, `xenium-xenium`, `xenium-starmap`, `visium-visium` and both
`heart` ones -- which differ from each other only in which table they read and how the
starting affine is obtained. `align_stalign_image` is the shape of the `merfish-visium` and
`xenium-heimage` ones. `align_landmarks` is `merfish-merfish-alignment-affine-only-with-points`.